In [ ]:
"""
Akkadian to English Translation Inference
Using My best ByT5 Optimized Model
"""

import re
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# CONFIGURATION
# ==============

INFERENCE_CONFIG = {
    # Path to test data
    "test_data_path": "/kaggle/input/deep-past-initiative-machine-translation/test.csv",
    
    # Path to trained model
    "trained_model_path": "/kaggle/input/final-byt5/byt5-akkadian-optimized-34x",
    
    # Device configuration
    "computation_device": torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    
    # Tokenization settings
    "maximum_sequence_length": 512,
    
    # Batch processing
    "inference_batch_size": 8,
    
    # Text generation parameters
    "generation_parameters": {
        "num_beams": 8,              # Number of beam search paths
        "max_new_tokens": 512,        # Maximum output length
        "length_penalty": 1.09,       # Penalty for longer sequences
        "early_stopping": True        # Stop when all beams finish
    }
}


# INPUT PREPROCESSING FUNCTIONS
# =============================

def preprocess_input_text(transliteration_text):
    """
    Clean and normalize input Akkadian transliteration
    
    Args:
        transliteration_text: Raw transliteration from dataset
        
    Returns:
        Cleaned transliteration with normalized gap markers
    """
    # Handle missing values
    if pd.isna(transliteration_text): 
        return ""
    
    # Convert to string
    cleaned_text = str(transliteration_text)
    
    # Normalize large gaps (multiple dots or ellipsis to <big_gap>)
    cleaned_text = re.sub(r'(\.{3,}|…+|……)', '<big_gap>', cleaned_text)
    
    # Normalize small gaps (x patterns to <gap>)
    cleaned_text = re.sub(r'(xx+|\s+x\s+)', '<gap>', cleaned_text)
    
    return cleaned_text


# OUTPUT POSTPROCESSING FUNCTIONS
# ===============================

def postprocess_translation_output(translation_text):
    """
    Clean and normalize model output translation
    
    Args:
        translation_text: Raw output from model
        
    Returns:
        Cleaned and formatted English translation
    """
    # Handle invalid outputs
    if not isinstance(translation_text, str) or not translation_text.strip(): 
        return ""
    
    processed_translation = translation_text
    
    # Step 1: Normalize special Akkadian characters
    processed_translation = processed_translation.replace('ḫ', 'h').replace('Ḫ', 'H')
    
    # Step 2: Convert subscript numbers to regular numbers
    subscript_to_normal = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
    processed_translation = processed_translation.translate(subscript_to_normal)
    
    # Step 3: Normalize gap markers in output
    processed_translation = re.sub(r'(\[x\]|\(x\)|\bx\b)', '<gap>', processed_translation, flags=re.I)
    processed_translation = re.sub(r'(\.{3,}|…|\[\.+\])', '<big_gap>', processed_translation)
    
    # Step 4: Merge adjacent gap markers
    processed_translation = re.sub(r'<gap>\s*<gap>', ' <big_gap> ', processed_translation)
    processed_translation = re.sub(r'<big_gap>\s*<big_gap>', ' <big_gap> ', processed_translation)
    
    # Step 5: Remove grammatical annotations
    processed_translation = re.sub(
        r'\((fem|plur|pl|sing|singular|plural|\?|!)\.?\s*\w*\)', 
        '', 
        processed_translation, 
        flags=re.I
    )
    
    # Step 6: Protect gap markers during character removal
    processed_translation = processed_translation.replace('<gap>', '\x00GAP\x00')
    processed_translation = processed_translation.replace('<big_gap>', '\x00BIG\x00')
    
    # Step 7: Remove unwanted punctuation and symbols
    forbidden_characters = '!?()"—–<>⌈⌋⌊[]+ʾ/;'
    processed_translation = processed_translation.translate(str.maketrans('', '', forbidden_characters))
    
    # Step 8: Restore gap markers
    processed_translation = processed_translation.replace('\x00GAP\x00', ' <gap> ')
    processed_translation = processed_translation.replace('\x00BIG\x00', ' <big_gap> ')
    
    # Step 9: Convert decimal fractions to Unicode fraction symbols
    fraction_patterns = {
        r'\.5\b': ' ½',
        r'\.25\b': ' ¼',
        r'\.75\b': ' ¾',
        r'\.33+\d*\b': ' ⅓',
        r'\.66+\d*\b': ' ⅔'
    }
    for decimal_pattern, fraction_symbol in fraction_patterns.items():
        processed_translation = re.sub(r'(\d+)' + decimal_pattern, r'\1' + fraction_symbol, processed_translation)
        processed_translation = re.sub(r'\b0' + decimal_pattern, fraction_symbol.strip(), processed_translation)
    
    # Step 10: Remove repeated words
    processed_translation = re.sub(r'\b(\w+)(?:\s+\1\b)+', r'\1', processed_translation)
    
    # Step 11: Remove repeated n-grams (phrases)
    for ngram_size in range(4, 1, -1):
        ngram_pattern = r'\b((?:\w+\s+){' + str(ngram_size-1) + r'}\w+)(?:\s+\1\b)+'
        processed_translation = re.sub(ngram_pattern, r'\1', processed_translation)
    
    # Step 12: Fix spacing around punctuation
    processed_translation = re.sub(r'\s+([.,:])', r'\1', processed_translation)
    processed_translation = re.sub(r'([.,])\1+', r'\1', processed_translation)
    
    # Step 13: Final whitespace normalization
    processed_translation = re.sub(r'\s+', ' ', processed_translation).strip().strip('-').strip()
    
    return processed_translation


# DATASET CLASS
# ==============

class AkkadianTranslationDataset(Dataset):
    """
    PyTorch Dataset for Akkadian transliterations
    """
    def __init__(self, dataframe):
        """
        Initialize dataset with transliteration data
        
        Args:
            dataframe: Pandas DataFrame with 'id' and 'transliteration' columns
        """
        self.sample_ids = dataframe['id'].tolist()
        
        # Add task prefix to each transliteration
        self.input_texts = [
            "translate Akkadian to English: " + str(transliteration) 
            for transliteration in dataframe['transliteration']
        ]
    
    def __len__(self):
        """Return total number of samples"""
        return len(self.sample_ids)
    
    def __getitem__(self, index):
        """
        Get single sample
        
        Args:
            index: Sample index
            
        Returns:
            Tuple of (sample_id, input_text)
        """
        return self.sample_ids[index], self.input_texts[index]


# MAIN INFERENCE PIPELINE
# ========================

# Load test data
print("Loading test data...")
test_dataframe = pd.read_csv(INFERENCE_CONFIG['test_data_path'])
test_dataframe['transliteration'] = test_dataframe['transliteration'].apply(preprocess_input_text)
print(f"Loaded {len(test_dataframe)} test samples")

# Load trained model
print("\nLoading trained model...")
translation_model = AutoModelForSeq2SeqLM.from_pretrained(
    INFERENCE_CONFIG['trained_model_path']
).to(INFERENCE_CONFIG['computation_device']).eval()

# Load tokenizer
model_tokenizer = AutoTokenizer.from_pretrained(INFERENCE_CONFIG['trained_model_path'])
print("Model and tokenizer loaded")

# Create dataset
inference_dataset = AkkadianTranslationDataset(test_dataframe)

# Create dataloader with custom collate function
def collate_batch_function(batch_samples):
    """
    Collate function to prepare batches
    
    Args:
        batch_samples: List of (id, text) tuples
        
    Returns:
        Tuple of (list_of_ids, tokenized_inputs)
    """
    batch_ids = [sample[0] for sample in batch_samples]
    batch_texts = [sample[1] for sample in batch_samples]
    
    # Tokenize all texts in batch
    tokenized_batch = model_tokenizer(
        batch_texts,
        max_length=INFERENCE_CONFIG['maximum_sequence_length'],
        padding=True,
        truncation=True,
        return_tensors="pt"
    )
    
    return batch_ids, tokenized_batch

inference_dataloader = DataLoader(
    inference_dataset,
    batch_size=INFERENCE_CONFIG['inference_batch_size'],
    shuffle=False,
    num_workers=2,
    collate_fn=collate_batch_function
)

print(f"\nDataloader created with {len(inference_dataloader)} batches")

# Run inference
print("\nRunning inference...")
prediction_results = []

with torch.inference_mode():
    for batch_ids, tokenized_inputs in inference_dataloader:
        # Generate translations
        generated_outputs = translation_model.generate(
            input_ids=tokenized_inputs.input_ids.to(INFERENCE_CONFIG['computation_device']),
            attention_mask=tokenized_inputs.attention_mask.to(INFERENCE_CONFIG['computation_device']),
            **INFERENCE_CONFIG['generation_parameters']
        )
        
        # Decode generated token IDs to text
        decoded_translations = model_tokenizer.batch_decode(
            generated_outputs, 
            skip_special_tokens=True
        )
        
        # Postprocess each translation
        cleaned_translations = [
            postprocess_translation_output(translation) 
            for translation in decoded_translations
        ]
        
        # Store results
        prediction_results.extend(zip(batch_ids, cleaned_translations))

print("Inference complete")

# Create submission file
print("\nCreating submission file...")
submission_dataframe = pd.DataFrame(
    prediction_results, 
    columns=['id', 'translation']
)

submission_dataframe.to_csv("submission.csv", index=False)
print(f"Submission saved with {len(submission_dataframe)} translations")